# __MODEL_LABEL_MARKDOWN__: model training

Load the source dataset, declare transforms and the model, then fit and
review the raw candidate. Save its version to the chosen database after
reviewing the metrics. The optional routine edit uses exported level groupings.


In [ ]:
DATABASE_MODE = __DATABASE_MODE_LITERAL__  # "local" or "remote"
RUNTIME_MODULE = __RUNTIME_MODULE_LITERAL__  # e.g. "project_runtime.database"; never put secrets here
EXPECTED_REMOTE_DATABASE = __EXPECTED_REMOTE_DATABASE_LITERAL__
ALLOW_REMOTE_WRITES = False

RECIPE_PATH = None  # None authors in Python; "challenger.toml" loads a file in MODEL_DIR.


In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "pricing_models").is_dir()
)

from superglm import Categorical, Numeric, SuperGLM

from pricing_pipeline.models.config import ValidationSplitConfig
from pricing_pipeline.notebook import (
    Clip,
    Log,
    Log1p,
    PricingDataset,
    PricingModelSpec,
    ModelRecipe,
    apply_level_groupings,
    apply_transforms,
    fit_model,
    connect,
    inspect_level_groupings,
    load_level_groupings,
    save_model_version,
    register_model,
)

MODEL_DIR = PROJECT_ROOT / "pricing_models/__PACKAGE_NAME__"
DATASET_PATH = MODEL_DIR / ".local" / "dataset.joblib"
GROUPING_ARTIFACT_PATH = MODEL_DIR / ".local" / "routine_groupings.joblib"


## Load the dataset


In [ ]:
pricing = connect(
    mode=DATABASE_MODE,
    runtime_module=RUNTIME_MODULE,
    local_root=MODEL_DIR / ".local",
    expected_remote_database=EXPECTED_REMOTE_DATABASE,
    allow_remote_writes=ALLOW_REMOTE_WRITES,
)
display(pricing.destination)


In [ ]:
# Data.
dataset = PricingDataset.load(DATASET_PATH)
df = dataset.df
display({"Rows": len(df), "Columns": len(df.columns)})


## Model

Set `RECIPE_PATH = "prototype.toml"` to load the configuration exported from 02.
Keep `RECIPE_PATH = None` to define the model in Python here.
Both choices use the shared fit and save cells below. Loading creates an unfitted
model with declared groups and special levels. Frozen learned knots or coefficients
still require the baseline artifact and monitoring presets.

Declare transforms once. Log requires positive inputs; Log1p computes log(1 + x).


In [ ]:
if RECIPE_PATH is None:
    # Transforms.
    transforms = {
        # "log_feature": Log("positive_feature"),
        # "log1p_feature": Log1p("nonnegative_feature"),
        # "clipped_feature": Clip("__FEATURE_NAME__", lower=0, upper=100),
        # "log_exposure": Log("exposure"),
    }

    # Features.
    RAW_FEATURES = {
        "__FEATURE_NAME__": Numeric(),
        "segment": Categorical(),
    }

    MODEL = PricingModelSpec(
        # Model.
        name="__MODEL_NAME__",
        label="__MODEL_LABEL__",
        model_type="__MODEL_TYPE__",
        deployment_slot="__DEPLOYMENT_SLOT__",

        # Data.
        dataset=dataset,

        # Fit and validate.
        target="__TARGET_NAME__",
        features=tuple(RAW_FEATURES),
        validation=ValidationSplitConfig.kfold(
            n_splits=5,
            random_state=42,
            shuffle=True,
        ),

        # Save the transforms with the rating tables.
        transforms=transforms,

        # Optional offset, with its coefficient fixed at 1.
        # offset_column="log_exposure",
        # Weights for fitting.
        # sample_weight_column="model_weight",
        # Weights for averaging exported rating tables.
        # export_weight_column="rating_table_weight",
    )

    raw_superglm_model = SuperGLM(
        family="poisson",
        selection_penalty=0.0,
        retain_fit_state=False,  # Drop training caches; keep prediction and summary.
        discrete=True,
        n_bins=64,
        features=RAW_FEATURES,
    )
else:
    MODEL, raw_superglm_model = ModelRecipe.load(MODEL_DIR / RECIPE_PATH).build(dataset=dataset)
    RAW_FEATURES = raw_superglm_model.features

# Data.
df = apply_transforms(dataset.df, MODEL.transforms)


## Raw model


In [ ]:
model = register_model(pricing, MODEL, source_root=MODEL_DIR)


## Fit and validate

`fit_model` fits the model on each validation split, refits the final
model on all rows, and exports its rating tables and fitted model. It also
writes the dataset manifest and split evidence to the chosen database.
Review the returned metrics before saving the version.

Pass a scikit-learn or custom splitter as `MODEL.validation` when you need
grouped or time-based CV. Set `groups_column` for grouped splits. Time-series
splitters use the saved dataframe order. Use column-based validation when the
dataset already contains split assignments.


In [ ]:
# Fit and validate.
raw_candidate = fit_model(
    pricing,
    model=model,
    frame=df,
    superglm_model=raw_superglm_model,
    model_kind="RAW",
)
raw_candidate.metrics


## Optional: export the raw recipe

Uncomment the call to save the exact configuration used for this fit. Existing
files require replace=True. SQL assigns revisions when a build is saved.


In [ ]:
# raw_candidate.recipe.save(MODEL_DIR / "raw_model.toml")


## Save the version

`save_model_version` saves this version to the chosen SQL Server or local
SQLite database. SQL Server versions are published; local versions are marked
`LOCAL_AUDIT`. Review the metrics above before running this cell. Activation
is a separate step in notebook 06.


In [ ]:
raw_published = save_model_version(pricing, raw_candidate)
display({
    "Model": raw_published.model_name,
    "Kind": raw_published.model_kind,
    "Package": raw_published.package_version,
    "Recipe": raw_published.recipe_revision,
    "Recipe status": raw_published.recipe_status,
    "Manifest": raw_published.manifest_id,
    "State": raw_published.package_status,
    "Reused equivalent": raw_published.deduplicated,
})


## Optional routine edit: existing grouping artifacts

Earlier projects may have exported `.local/routine_groupings.joblib` from a
published RAW model. This section loads those groupings and skips the routine
fit when the artifact is absent or contains no collapse.

A recipe from 02 already contains your chosen groupings. Recipe mode skips this
artifact; change the feature configuration explicitly when trying new groupings.


In [ ]:
LEVEL_GROUPINGS = (
    load_level_groupings(
        GROUPING_ARTIFACT_PATH,
        frame=df,
        model=model,
    )
    if RECIPE_PATH is None and GROUPING_ARTIFACT_PATH.is_file()
    else {}
)
ROUTINE_EDIT_CONFIGURED = bool(LEVEL_GROUPINGS)
routine_superglm_model = None
if ROUTINE_EDIT_CONFIGURED:
    ROUTINE_FEATURES = apply_level_groupings(RAW_FEATURES, LEVEL_GROUPINGS)
    routine_superglm_model = SuperGLM(
        family="poisson",
        selection_penalty=0.0,
        retain_fit_state=False,  # Drop training caches; keep prediction and summary.
        discrete=True,
        n_bins=64,
        features=ROUTINE_FEATURES,
    )
    display(inspect_level_groupings(GROUPING_ARTIFACT_PATH))
else:
    display("No exported level collapses: ROUTINE_EDIT skipped.")


`fit_model` fits and cross-validates the grouped model, refits on all
rows, exports its rating tables, and writes dataset and split evidence to the
chosen database. This cell then saves the routine version there. SQL Server
publication and local SQLite audit storage do not activate the model.


In [ ]:
routine_published = None
if ROUTINE_EDIT_CONFIGURED:
    routine_candidate = fit_model(
        pricing,
        model=model,
        frame=df,
        superglm_model=routine_superglm_model,
        model_kind="ROUTINE_EDIT",
    )
    display(routine_candidate.metrics)
    routine_published = save_model_version(pricing, routine_candidate)
    display({
        "Kind": routine_published.model_kind,
        "Package": routine_published.package_version,
        "Recipe": routine_published.recipe_revision,
        "Manifest": routine_published.manifest_id,
        "State": routine_published.package_status,
        "Reused equivalent": routine_published.deduplicated,
    })


## Optional: export the routine recipe

The routine candidate contains its applied groupings. The raw recipe describes
the raw fit. Keep separate files when comparing them.


In [ ]:
# if ROUTINE_EDIT_CONFIGURED:
#     routine_candidate.recipe.save(MODEL_DIR / "routine_model.toml")
